In [1]:
! pip install mistralai

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached googleapis_common_protos-1.72.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp313-cp313-win_amd64.whl.metadata (7.4 kB)
  Using cache


[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import base64
import requests
import os
from mistralai import Mistral

In [5]:
def encode_pdf(pdf_path):
    """Encode the pdf to base64."""
    try:
        with open(pdf_path, "rb") as pdf_file:
            return base64.b64encode(pdf_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {pdf_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None


In [6]:
# Path to your pdf
pdf_path = "../input_files/Sample Contract.pdf"

# Getting the base64 string
base64_pdf = encode_pdf(pdf_path)

In [ ]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("MISTRAL_API_KEY")


'WzC7Z9sSdFWNHRcGC12d5XIpyeTGRYE7'

In [8]:
import os
from mistralai import Mistral

client = Mistral(api_key=api_key)

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": f"data:application/pdf;base64,{base64_pdf}"
    },
    table_format="html", # default is None
    # extract_header=True, # default is False
    # extract_footer=True, # default is False
    include_image_base64=True
)

In [9]:

from pathlib import Path

pdf_file = Path("../input_files/Sample Contract.pdf")
assert pdf_file.is_file()

In [10]:
from mistralai import DocumentURLChunk, ImageURLChunk, TextChunk
import json

uploaded_file = client.files.upload(
    file={
        "file_name": pdf_file.stem,
        "content": pdf_file.read_bytes(),
    },
    purpose="ocr",
)

signed_url = client.files.get_signed_url(file_id=uploaded_file.id, expiry=1)

pdf_response = client.ocr.process(document=DocumentURLChunk(document_url=signed_url.url),
                                  model="mistral-ocr-latest",
                                  include_image_base64=True)

response_dict = json.loads(pdf_response.json())
json_string = json.dumps(response_dict, indent=4)

print(json_string)

{
    "pages": [
        {
            "index": 0,
            "markdown": "# Information Security and Technology Risk Addendum\n\nThis Information Security and Technology Risk Addendum (this \"Addendum\") is entered into as of January 15, 2026 (the \"Addendum Effective Date\") by and between Redwood Peak Financial, Inc., a Delaware corporation (\"Company\"), and Nimbus Ridge Technologies, LLC, a California limited liability company (\"Vendor\"). This Addendum is incorporated into and forms part of the Master Services Agreement dated January 10, 2026 (the \"Agreement\"). Capitalized terms not defined in this Addendum have the meanings given in the Agreement.\n\n## 1. Order of Precedence\n\nIf there is a conflict between this Addendum and the Agreement relating to information security, confidentiality, privacy, audit, incident response, or audit/assurance deliverables, this Addendum controls for that subject matter.\n\n## 2. Scope and Covered Environments\n\n2.1 Scope. This Addendum app

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_28752\850618116.py:18: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  response_dict = json.loads(pdf_response.json())


In [8]:
from mistralai.models import OCRResponse
from IPython.display import Markdown, display

def replace_images_in_markdown(markdown_str: str, images_dict: dict) -> str:
    for img_name, base64_str in images_dict.items():
        markdown_str = markdown_str.replace(f"![{img_name}]({img_name})", f"![{img_name}]({base64_str})")
    return markdown_str

def get_combined_markdown(ocr_response: OCRResponse) -> str:
  markdowns: list[str] = []
  for page in pdf_response.pages:
    image_data = {}
    for img in page.images:
      image_data[img.id] = img.image_base64
    markdowns.append(replace_images_in_markdown(page.markdown, image_data))
  
  return "\n\n".join(markdowns)

# display(Markdown(get_combined_markdown(pdf_response)))

In [12]:
from mistralai.models import OCRResponse
from IPython.display import Markdown, display

def replace_images_in_markdown(markdown_str: str, images_dict: dict) -> str:
    for img_name, base64_str in images_dict.items():
        markdown_str = markdown_str.replace(f"![{img_name}]({img_name})", f"![{img_name}]({base64_str})")
    return markdown_str

def get_combined_markdown(ocr_response: OCRResponse) -> str:
  markdowns: list[str] = []
  for page in pdf_response.pages:
    image_data = {}
    for img in page.images:
      image_data[img.id] = img.image_base64
    markdowns.append(replace_images_in_markdown(page.markdown, image_data))
  
  return "\n\n".join(markdowns)

markdown_output = get_combined_markdown(pdf_response)


In [ ]:
with open("../MarkdownFiles/ocr_output.md", "w", encoding="utf-8") as f:
    f.write(markdown_output)

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import json

def preprocess_contract_markdown(raw_markdown):
    # 1. Dynamically define headers to look for (Standard Markdown levels)
    # This will catch #, ##, ###, #### regardless of what they are named
    headers_to_split_on = [
        ("#", "Level_1"),
        ("##", "Level_2"),
        ("###", "Level_3"),
        ("####", "Level_4"),
    ]

    # 2. Initialize the dynamic splitter
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, 
        strip_headers=False # Keep the headers in the text so the LLM sees them too
    )

    # 3. Split based on the structure Mistral OCR found
    md_header_splits = markdown_splitter.split_text(raw_markdown)

    # 4. Refine with a character splitter 
    # This ensures that if a section is massive, it's broken down for the LLM
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,  # Balanced for context and precision
        chunk_overlap=200
    )
    
    final_chunks = text_splitter.split_documents(md_header_splits)

    # 5. Build the Dynamic JSON Object
    processed_data = []
    for i, chunk in enumerate(final_chunks):
        # Create a breadcrumb trail from whatever headers were found
        # e.g., "Addendum > Section 6 > 6.2 MFA"
        breadcrumb = " > ".join([
            chunk.metadata.get(f"Level_{j}", "") 
            for j in range(1, 5) if f"Level_{j}" in chunk.metadata
        ])

        processed_data.append({
            "block_id": f"chunk_{i}",
            "title": breadcrumb if breadcrumb else "General Content",
            "content": chunk.page_content,
            "metadata": {
                **chunk.metadata,
                "chunk_index": i
            }
        })
    
    return processed_data

# Example Usage:
# result = preprocess_contract_markdown(your_ocr_output_string)

In [3]:
! pip install langchain-text-splitters

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached zstandard-0.25.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
Using cached jsonpointer-3.0.0-py2.py3-none-any.whl (7.6 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl (31 kB)
Using cached zstandard-0.25.0-cp313-cp313-win_amd64.whl (506 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
import re

def contract_to_main_sections(md_text):
    # 1. Split only on Main Section Headers (Level 1 in your MD)
    # This keeps "2. Scope" and all its tables (2.1, 2.2, 2.3) in one row.
    headers_to_split_on = [
        ("#", "section_title"),
    ]

    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    sections = splitter.split_text(md_text)

    processed_rows = []
    
    for i, doc in enumerate(sections):
        raw_header = doc.metadata.get("section_title", "General")
        content = doc.page_content
        
        # 2. Dynamic ID Extraction (e.g., "2. Scope" -> "2")
        id_match = re.match(r"(\d+)\.?\s*(.*)", raw_header)
        section_id = id_match.group(1) if id_match else str(i)
        clean_title = id_match.group(2) if id_match else raw_header

        # 3. Build the Row
        # Combining title and content ensures the RAG embedding captures the topic.
        row = {
            "block_id": section_id,
            "title": clean_title,
            "level": 1,
            "content": f"{raw_header}\n{content}",
            "parent_section": section_id,
            "block_type": "legal_clause",
            "page_numbers": [int(m.group(1)) for m in re.finditer(r"--- PAGE (\d+) ---", content)],
            "keywords": auto_tag(content)
        }
        processed_rows.append(row)
        
    return processed_rows

def auto_tag(text):
    mapping = {"mfa": "MFA", "encrypt": "Encryption", "scope": "Scope", "asset": "Assets"}
    return [v for k, v in mapping.items() if k in text.lower()]

In [7]:
with open("../MarkdownFiles/ocr_output.md", "r", encoding="utf-8") as f:
    content = f.read()
sections = contract_to_main_sections(content)

In [8]:
sections

[{'block_id': '0',
  'title': 'Information Security and Technology Risk Addendum',
  'level': 1,
  'content': 'Information Security and Technology Risk Addendum\nThis Information Security and Technology Risk Addendum (this "Addendum") is entered into as of January 15, 2026 (the "Addendum Effective Date") by and between Redwood Peak Financial, Inc., a Delaware corporation ("Company"), and Nimbus Ridge Technologies, LLC, a California limited liability company ("Vendor"). This Addendum is incorporated into and forms part of the Master Services Agreement dated January 10, 2026 (the "Agreement"). Capitalized terms not defined in this Addendum have the meanings given in the Agreement.  \n## 1. Order of Precedence  \nIf there is a conflict between this Addendum and the Agreement relating to information security, confidentiality, privacy, audit, incident response, or audit/assurance deliverables, this Addendum controls for that subject matter.  \n## 2. Scope and Covered Environments  \n2.1 Sco